In [ ]:
from typing import List, Tuple
from pre_processing_tokenizer import tokenize_raw_text

def check_conll_consistency(conll_file_path: str):
    """
    Reads a CoNLL file and checks if the tokenization of the annotated data
    is consistent with the pre_processing_tokenizer's logic.
    
    Args:
        conll_file_path: The path to the CoNLL file.
    """
    inconsistencies = []
    sentence_tokens = []
    sentence_tags = []
    sentence_number = 0
    
    with open(conll_file_path, 'r', encoding='utf-8') as f:
        for line_number, line in enumerate(f, 1):
            line = line.strip()
            if line:
                parts = line.split('\t')
                if len(parts) == 2:
                    token, tag = parts
                    sentence_tokens.append(token)
                    sentence_tags.append(tag)
                else:
                    # Handle malformed lines if any
                    continue
            else:
                # Blank line indicates end of sentence
                if sentence_tokens:
                    sentence_number += 1
                    raw_text = " ".join(sentence_tokens)
                    re_tokenized = tokenize_raw_text(raw_text)

                    # Simple check for same number of tokens
                    if len(sentence_tokens) != len(re_tokenized):
                        inconsistencies.append({
                            "sentence_number": sentence_number,
                            "coached_line_numbers": (line_number - len(sentence_tokens), line_number),
                            "reason": "Token count mismatch.",
                            "original_tokens": sentence_tokens,
                            "re_tokenized": re_tokenized
                        })
                    else:
                        # More detailed token-by-token comparison
                        for i in range(len(sentence_tokens)):
                            if sentence_tokens[i] != re_tokenized[i] and not (sentence_tokens[i].startswith("##")):
                                inconsistencies.append({
                                    "sentence_number": sentence_number,
                                    "reason": "Token mismatch.",
                                    "original_token": sentence_tokens[i],
                                    "re_tokenized_token": re_tokenized[i]
                                })
                                break

                    sentence_tokens = []
                    sentence_tags = []
    
    # Report findings
    if inconsistencies:
        print("Found inconsistencies in tokenization!")
        for issue in inconsistencies:
            print("-" * 50)
            print(f"Sentence {issue['sentence_number']}:")
            print(f"Reason: {issue['reason']}")
            if 'original_tokens' in issue:
                print(f"Original tokens: {issue['original_tokens']}")
                print(f"Re-tokenized: {issue['re_tokenized']}")
            else:
                print(f"Original token: '{issue['original_token']}'")
                print(f"Re-tokenized token: '{issue['re_tokenized_token']}'")
    else:
        print("CoNLL file is consistent with the pre-processing tokenizer!")

# --- Usage ---
# Ensure your CoNLL file is in the same directory.
conll_file = 'annotated_data.conll'
check_conll_consistency(conll_file)